# Scene classification: Kaggle training runner

Settings before running: **Accelerator = GPU T4**, **Internet = On**, and add the input dataset **Intel Image Classification** (puneet6060).

This notebook holds no model code. It clones the GitHub repo and runs the scripts, so every run maps to a git commit.

In [ ]:
# 1. Confirm the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Find the dataset (Kaggle mount paths vary)
import glob
train_dir = glob.glob('/kaggle/input/**/seg_train/seg_train', recursive=True)[0]
test_dir = glob.glob('/kaggle/input/**/seg_test/seg_test', recursive=True)[0]
print(train_dir, test_dir, sep='\n')

In [ ]:
# 3. Get the code. Replace <you> with your GitHub username
REPO = 'https://github.com/dhruveleven/scene_classification.git'
!rm -rf scene_classification && git clone -q $REPO
%cd scene_classification
!git log -1 --oneline

In [ ]:
# 4. Quick interactive check: 1 epoch on 2000 images (~1 min)
!python train.py --config configs/kaggle.yaml --train_dir $train_dir --limit 2000 --epochs 1 --out_dir /kaggle/working/runs/smoke

In [ ]:
# 4b. README smoke test (local.yaml settings, 200 images)
!python train.py --config configs/local.yaml --train_dir $train_dir --limit 200 --out_dir /kaggle/working/runs/smoke_local

In [ ]:
# 4c. Overfit check: 32 images, 30 epochs, no freeze. Train loss should approach the label-smoothing floor (~0.42 for 6 classes, ls=0.1)
!python train.py --config configs/local.yaml --train_dir $train_dir --limit 32 --epochs 30 --freeze_epochs 0 --out_dir /kaggle/working/runs/overfit

When step 4 works: **Save Version > Save & Run All (Commit)** to run the full training in the background. You can close the tab.

In [ ]:
# 5. Full training run
!python train.py --config configs/kaggle.yaml --train_dir $train_dir

In [ ]:
# 6. Final test-set evaluation (run once, after you stop tuning)
!python evaluate.py --ckpt /kaggle/working/runs/exp1/best.pt --test_dir $test_dir | tee /kaggle/working/runs/exp1/test_report.txt

In [ ]:
# 6b. Export ONNX (done here because torch is blocked on the laptop)
!pip install -q onnx onnxscript
!python export_onnx.py --ckpt /kaggle/working/runs/exp1/best.pt --out /kaggle/working/model.onnx
!cp /kaggle/working/runs/exp1/best.pt /kaggle/working/best.pt
!ls -lh /kaggle/working

In [ ]:
# 7. Clean up: keep metrics.csv/config.yaml from the check runs, drop their checkpoints
!rm -rf /kaggle/working/runs/smoke
!rm -f /kaggle/working/runs/smoke_local/*.pt /kaggle/working/runs/overfit/*.pt
!ls -lhR /kaggle/working/runs

## Resuming after a timeout
1. In a new version, **Add Input > Notebook Output** and pick the previous version.
2. Copy the checkpoint, then resume:
```
!mkdir -p /kaggle/working/runs/exp1
!cp /kaggle/input/<prev-notebook>/runs/exp1/* /kaggle/working/runs/exp1/
!python train.py --config configs/kaggle.yaml --train_dir $train_dir --resume
```